# Live2Dパーツ補完 — 起動ノートブック

マシンを起動するたびに、**上から順に実行**してください(メニューの「Run All」でも可)。
モデルは一時領域に置くので、マシンを止めると消えます。このノートブックを実行し直せば元に戻ります。

| セル | 内容 | 目安 |
|---|---|---|
| 1 | 設定 | すぐ |
| 2 | Hugging Faceトークンの登録(**初回だけ**) | すぐ |
| 3 | ComfyUIと部品の準備 | 1〜3分 |
| 4 | モデルの取得 | 5〜10分 |
| 5 | ComfyUIの起動(実行したままにする) | 1分 |

## 1. 設定
使わないモデルは `False` にすると、その分の取得を省けます。

In [ ]:
USE_QWEN = True     # Qwen-Image-Edit-2511 (約29GB / トークン不要)
USE_FLUX = True     # FLUX 3 Image Klein   (約24GB / 初回だけトークン登録が必要)

# True にするとモデルを /notebooks/models に保存し、次回から取得を省略します。
# ただし保存容量を 50GB 以上使います(契約プランによっては追加料金の対象)。
KEEP_MODELS = False

import os, pathlib
REPO_URL  = 'https://github.com/yumi-ai-lab/paperspace-comfyui-live2d-parts'
REPO_DIR  = pathlib.Path('/notebooks/live2d-parts-repo')
TOKEN_PATH = pathlib.Path('/notebooks/.hf/token')
os.environ['MODEL_DIR'] = '/notebooks/models' if KEEP_MODELS else '/tmp/models'
os.environ['HF_TOKEN_PATH'] = str(TOKEN_PATH)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('モデルの置き場所:', os.environ['MODEL_DIR'])

## 2. Hugging Faceトークンの登録(初回だけ)

FLUXを使う場合だけ必要です。事前に次の2つを済ませてください。

1. https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy でライセンスに同意
2. https://huggingface.co/settings/tokens で読み取り用(Read)トークンを作成

入力したトークンは画面に表示されず、あなたの保存領域(`/notebooks/.hf/token`)にだけ保存されます。
**2回目以降は入力不要**です。消したいときはそのファイルを削除してください。

In [ ]:
import getpass
if not USE_FLUX:
    print('FLUXを使わない設定なので、トークンは不要です。')
elif TOKEN_PATH.exists() and TOKEN_PATH.read_text().strip():
    print('登録済みのトークンを使います。')
else:
    t = getpass.getpass('Hugging Faceのトークンを貼り付けてEnter: ').strip()
    if not t.startswith('hf_'):
        raise SystemExit('トークンは hf_ で始まります。もう一度このセルを実行してください。')
    TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)
    TOKEN_PATH.write_text(t); TOKEN_PATH.chmod(0o600)
    print('保存しました。次回からは入力不要です。')

## 3. ComfyUIと部品の準備
最新のスクリプトとワークフローを取り込み、ComfyUI本体とSee-throughノードを用意します。

In [ ]:
import subprocess
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
!bash {REPO_DIR}/scripts/setup.sh 2>&1 | tail -n 3

## 4. モデルの取得
すでに取得済みのファイルは飛ばします。途中で止まっても、このセルを実行し直せば続きから再開します。

In [ ]:
if USE_QWEN:
    !bash {REPO_DIR}/scripts/download_models.sh qwen 2>&1 | tail -n 8
if USE_FLUX:
    !bash {REPO_DIR}/scripts/download_models.sh flux 2>&1 | tail -n 8
!du -sh $MODEL_DIR

## 5. ComfyUIの起動

このセルは**実行したまま**にしてください(止めるとComfyUIも止まります)。
実行中のセルがあると、操作がなくてもマシンが自動停止されにくくなります。

表示されたURLをブラウザで開くとComfyUIの画面になります。ワークフローは左メニューの `live2d-parts` フォルダにあります。

In [ ]:
fqdn = os.environ.get('PAPERSPACE_FQDN', '')
print('ComfyUIのURL:', f'https://{fqdn}/proxy/8188/' if fqdn else '(このノートブックのURLの末尾を /proxy/8188/ に変えたもの)')
print('動作テスト(別のターミナルで):', f'python {REPO_DIR}/scripts/run_test.py {REPO_DIR}/workflows/live2d_inpaint_flux3klein_distilled_noupsampler.json')
!bash {REPO_DIR}/scripts/start.sh